In [0]:
%sql
create or replace temp view vw_gold_claims_analysis
as
select
   policy_type,
   
    avg(claim_amount) as avg_claim_amount ,
    min(claim_amount) as min_claim_amount ,
    max(claim_amount) as max_claim_amount ,
    count(DISTINCT claim_id) as total_claim
   
FROM
   silverlayer.claim c
   Join silverlayer.policydata p
   on c.policy_id = p.policy_id
Group by
   policy_type having policy_type is not null;

In [0]:
%sql
select * from vw_gold_claims_analysis

policy_type,avg_claim_amount,min_claim_amount,max_claim_amount,total_claim
Auto,99835.5613,2283,197403,351
Health,97129.2629,2188,199995,350


In [0]:
%sql
MERGE INTO goldlayer.claims_analysis as T USING vw_gold_claims_analysis as s on 
t.policy_type = s.policy_type  WHEN MATCHED THEN UPDATE SET t.total_claim =s.total_claim,
t.avg_claim_amount = s.avg_claim_amount, t.min_claim_amount = s.min_claim_amount, t.max_claim_amount = s.max_claim_amount, t.update_timestamp = current_timestamp()
WHEN NOT MATCHED THEN INSERT (policy_type, total_claim,avg_claim_amount, min_claim_amount, max_claim_amount, update_timestamp)
values(s.policy_type, s.total_claim, s.avg_claim_amount,s.max_claim_amount,s.min_claim_amount, current_timestamp())

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
select * from goldlayer.claims_analysis

policy_type,total_claim,avg_claim_amount,min_claim_amount,max_claim_amount,update_timestamp
Auto,351.0,99835.5613,197403.0,2283.0,2026-10-25T17:08:12.292Z
Health,350.0,97129.2629,199995.0,2188.0,2026-10-25T17:08:12.292Z
